In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
from zipfile import ZipFile
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# --- Steps 1 & 2: Unzip and find the file (no changes) ---
zip_file_path = 'archive.zip'
extract_folder = 'ohio-dataset'
if not os.path.exists(extract_folder):
    os.makedirs(extract_folder)
with ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)
print(f"Files extracted to '{extract_folder}' folder.")

xml_file_to_read = None
target_filename = '559-ws-training.xml'
for root_dir, dirs, files in os.walk(extract_folder):
    if target_filename in files:
        xml_file_to_read = os.path.join(root_dir, target_filename)
        break
if xml_file_to_read:
    print(f"\nFound the file at: {xml_file_to_read}")
else:
    print(f"\nERROR: Could not find '{target_filename}'.")
    exit()

# --- Step 3: Parse the XML file ---
tree = ET.parse(xml_file_to_read)
root = tree.getroot()
data_list = []

# Modify parsing logic to get event type from parent tag
for parent in root.findall('./*'): # Iterate through direct children of root
    event_type = parent.tag # Get the parent tag name
    for event in parent.findall('.//event'): # Find all event tags within the parent
        data_list.append({
            'timestamp': event.get('ts'),
            'event_type': event_type, # Use the parent tag name as event_type
            'value': event.get('value')
        })

print("\n--- DEBUG STEP 0.4: First 5 entries of data_list ---")
print(data_list[:5])


# --- Step 4 & 5: Create, Clean, and Pivot DataFrame (no changes) ---
df = pd.DataFrame(data_list)
if df.empty:
    print("\nERROR: DataFrame is empty. No <event> tags found with required attributes.")
    exit()
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%m-%Y %H:%M:%S')
df['value'] = pd.to_numeric(df['value'], errors='coerce')

print("\n--- DEBUG STEP 0.5: event_type value counts ---")
print(df['event_type'].value_counts())

df_pivot = df.pivot_table(index='timestamp', columns='event_type', values='value', aggfunc='first')

print("\n--- DEBUG STEP 0.6: df_pivot before renaming ---")
print(df_pivot.head())

# --- Step 6: Process and DEBUG the Pivoted DataFrame ---
rename_dict = {'glucose_level': 'glucose', 'meal': 'carbs', 'bolus': 'insulin'} # Updated rename_dict
df_pivot.rename(columns=rename_dict, inplace=True)

print("\n--- DEBUG STEP 0.7: df_pivot after renaming ---")
print(df_pivot.head())


for col in ['glucose', 'carbs', 'insulin']:
    if col not in df_pivot.columns:
        df_pivot[col] = np.nan

df_final = df_pivot.resample('5min').mean()

print("\n--- DEBUG STEP 1: Statistics AFTER Resampling ---")
print("This shows the raw data stats. 'count' should be > 0 for glucose.")
print(df_final['glucose'].describe())

# --- Step 7: Clean and Impute Data ---
df_final['glucose'].interpolate(method='linear', inplace=True)
df_final['carbs'].fillna(0, inplace=True)
df_final['insulin'].fillna(0, inplace=True)
df_final.dropna(subset=['glucose'], inplace=True) # Drop if any NaNs remain at start/end


print("\n--- DEBUG STEP 2: Statistics AFTER Cleaning & Interpolating ---")
print("The 'count' should increase, and 'mean'/'std' should be reasonable glucose values.")
print(df_final['glucose'].describe())


# --- Step 8: Normalize the Data (The NEXT MAJOR STEP) ---
# We scale the data to be between 0 and 1. This is required for deep learning.
scaler = MinMaxScaler()
df_final[['glucose', 'carbs', 'insulin']] = scaler.fit_transform(df_final[['glucose', 'carbs', 'insulin']])

print("\n--- DEBUG STEP 3: Statistics AFTER Normalizing ---")
print("After scaling, 'min' should be 0.0 and 'max' should be 1.0.")
print(df_final['glucose'].describe())

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(df_final.index, df_final['glucose'], label='Normalized Glucose Data', color='purple')
plt.title('Patient Glucose Levels Over Time (Normalized)')
plt.xlabel('Timestamp')
plt.ylabel('Normalized Glucose (0 to 1)')
plt.legend()
plt.grid(True)
plt.show()

print("\nFinal Processed DataFrame Head:")
print(df_final.head())

In [ ]:
# Read the XML file content to inspect its structure
xml_file_path = 'ohio-dataset/559-ws-training.xml' # Use the path found earlier

try:
    with open(xml_file_path, 'r') as f:
        xml_content = f.read()
    print("--- Beginning of the XML file content ---")
    print(xml_content[:1000]) # Print the first 1000 characters to inspect the structure
except FileNotFoundError:
    print(f"Error: XML file not found at {xml_file_path}")

In [ ]:
# --- Step 10: Create Time-Series Windows (Sequences) ---

print("\n--- Creating Time-Series Sequences ---")

# Define the features we will use as input to the model
# For now, we'll use the three core features.
features = ['glucose', 'carbs', 'insulin']
target = ['glucose']

# Define the window sizes
INPUT_WINDOW = 24  # Use the last 24 steps (24 * 5 mins = 2 hours) of data as input
OUTPUT_WINDOW = 6  # Predict the next 6 steps (6 * 5 mins = 30 mins) of glucose

# Create empty lists to store the sequences
X = []
y = []

# Convert the relevant columns to a NumPy array for efficiency
data_np = df_final[features].values
target_np = df_final[target].values

for i in range(len(data_np) - INPUT_WINDOW - OUTPUT_WINDOW + 1):
    # The input window is a chunk of 24 time steps
    input_window = data_np[i : i + INPUT_WINDOW]

    # The output window is the 6 glucose values that come after the input window
    output_window = target_np[i + INPUT_WINDOW : i + INPUT_WINDOW + OUTPUT_WINDOW]

    X.append(input_window)
    y.append(output_window)

# Convert the lists to NumPy arrays
X = np.array(X)
y = np.array(y)

print(f"Shape of the input sequences (X): {X.shape}")
print(f"Shape of the output sequences (y): {y.shape}")
# Expected output: (number of samples, 24 timesteps, 3 features)
# Expected output: (number of samples, 6 timesteps, 1 feature)


# --- Step 11: Split the Data into Training, Validation, and Test Sets ---
# It is crucial to split time-series data chronologically.

print("\n--- Splitting the Data ---")

# Calculate split indices
train_split = int(len(X) * 0.7)
val_split = int(len(X) * 0.85) # 70% for train, 15% for val, 15% for test

# Perform the split
X_train, y_train = X[:train_split], y[:train_split]
X_val, y_val = X[train_split:val_split], y[train_split:val_split]
X_test, y_test = X[val_split:], y[val_split:]

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, LayerNormalization, Dropout

class MultiHeadSelfAttention(Layer):
    def __init__(self, embed_dim, num_heads=8):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.projection_dim = embed_dim // num_heads
        self.query_dense = Dense(embed_dim)
        self.key_dense = Dense(embed_dim)
        self.value_dense = Dense(embed_dim)
        self.combine_heads = Dense(embed_dim)


    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights


    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])


    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)
        attention, _ = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output

class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)


    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TransformerEncoder(Layer):
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerEncoder, self).__init__()
        self.num_layers = num_layers
        self.embed_dim = embed_dim
        self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate) for _ in range(num_layers)]
        self.dropout = Dropout(rate)

    def call(self, inputs, training=False):
        x = inputs
        for i in range(self.num_layers):
            x = self.enc_layers[i](x, training=training)
        return x



In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Dense, LayerNormalization, Dropout, Input, Flatten
class PositionalEncoding(Layer):
    def __init__(self, position, embed_dim):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(position, embed_dim)

    def get_angles(self, pos, i, embed_dim):
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(embed_dim))
        return pos * angle_rates

    def positional_encoding(self, position, embed_dim):
        angle_rads = self.get_angles(np.arange(position)[:, np.newaxis],
                                     np.arange(embed_dim)[np.newaxis, :],
                                     embed_dim)
        # apply sin to even indices in the array; 2i
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        # apply cos to odd indices in the array; 2i+1
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(pos_encoding, dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]


In [ ]:
# Model Hyperparameters
NUM_LAYERS = 4
EMBED_DIM = 64  # Embedding dimension
NUM_HEADS = 4   # Number of attention heads
FF_DIM = 128    # Hidden layer size in feed forward network inside transformer
DROPOUT_RATE = 0.1
INPUT_SHAPE = (X_train.shape[1], X_train.shape[2]) # (24, 3)
OUTPUT_SHAPE = (y_train.shape[1], y_train.shape[2]) # (6, 1)

# 1. Input Layer
inputs = Input(shape=INPUT_SHAPE)

# 2. Project input features to the embedding dimension
# This is a common practice for time-series transformers
dense_input = Dense(EMBED_DIM)(inputs)

# 3. Add Positional Encoding
pos_encoding_layer = PositionalEncoding(INPUT_SHAPE[0], EMBED_DIM)
x = pos_encoding_layer(dense_input)

# 4. Your Transformer Encoder
encoder = TransformerEncoder(NUM_LAYERS, EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT_RATE)
x = encoder(x)

# 5. Output Head for Prediction
# Flatten the sequence and then project to the desired output shape
x = Flatten()(x)
x = Dense(FF_DIM, activation="relu")(x)
outputs = Dense(OUTPUT_SHAPE[0] * OUTPUT_SHAPE[1])(x)
# Reshape the flat output to the desired sequence shape (6, 1)
outputs = tf.keras.layers.Reshape(OUTPUT_SHAPE)(outputs)

# Create the final Keras model
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(optimizer="adam", loss="mse", metrics=['mae'])

# Print the model summary
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)


In [ ]:
print("\n--- Evaluating Model on Unseen Test Data ---")
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test Loss (MSE): {test_loss:.4f}")
print(f"Test Mean Absolute Error (MAE): {test_mae:.4f}")


In [ ]:
print("\n--- Making Predictions ---")
predictions_normalized = model.predict(X_test)

In [ ]:
predictions_reshaped = predictions_normalized.reshape(-1, OUTPUT_WINDOW)

# Create a dummy array for predictions
dummy_predictions = np.zeros((predictions_reshaped.shape[0], len(features)))
dummy_predictions[:, 0] = predictions_reshaped[:, 0] # Use the first predicted step for simplicity-
predictions_mg_dl = scaler.inverse_transform(dummy_predictions)[:, 0]

# Reshape true values (y_test) to 2D for the scaler
y_test_reshaped = y_test.reshape(-1, OUTPUT_WINDOW)

# Create a dummy array for true values
dummy_y_test = np.zeros((y_test_reshaped.shape[0], len(features)))
dummy_y_test[:, 0] = y_test_reshaped[:, 0]
y_test_mg_dl = scaler.inverse_transform(dummy_y_test)[:, 0]

In [ ]:
print("\n--- Visualizing Predictions vs. True Values ---")

plt.figure(figsize=(15, 6))
plt.plot(y_test_mg_dl, color='blue', label='True Glucose Values')
plt.plot(predictions_mg_dl, color='red', label='Predicted Glucose Values', alpha=0.7)
plt.title('Glucose Prediction vs. Actual Values (Test Set)')
plt.xlabel('Time Steps (in 5-minute intervals)')
plt.ylabel('Glucose (mg/dL)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Define the LSTM model
lstm_model = Sequential()

# Add an LSTM layer with a return_sequences=True to pass the sequence to the next layer
# You can experiment with the number of units
lstm_model.add(LSTM(units=64, return_sequences=True, input_shape=INPUT_SHAPE))
lstm_model.add(Dropout(0.1))

# Add another LSTM layer
lstm_model.add(LSTM(units=64))
lstm_model.add(Dropout(0.1))

# Add a Dense layer to project the LSTM output to the desired output shape
lstm_model.add(Dense(OUTPUT_SHAPE[0] * OUTPUT_SHAPE[1]))

# Reshape the output to the desired sequence shape (6, 1)
lstm_model.add(tf.keras.layers.Reshape(OUTPUT_SHAPE))


# Compile the model
lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Print the model summary
lstm_model.summary()

In [ ]:
# Train the LSTM model
print("\n--- Training the LSTM Model ---")

early_stopping_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

lstm_model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping_lstm]
)

In [ ]:
# Evaluate the LSTM model
print("\n--- Evaluating LSTM Model on Unseen Test Data ---")
lstm_test_loss, lstm_test_mae = lstm_model.evaluate(X_test, y_test)
print(f"LSTM Test Loss (MSE): {lstm_test_loss:.4f}")
print(f"LSTM Test Mean Absolute Error (MAE): {lstm_test_mae:.4f}")

In [ ]:
# Make predictions with the LSTM model
print("\n--- Making Predictions with LSTM Model ---")
lstm_predictions_normalized = lstm_model.predict(X_test)

# Inverse transform the predictions
lstm_predictions_reshaped = lstm_predictions_normalized.reshape(-1, OUTPUT_WINDOW)

# Create a dummy array for predictions
dummy_lstm_predictions = np.zeros((lstm_predictions_reshaped.shape[0], len(features)))
dummy_lstm_predictions[:, 0] = lstm_predictions_reshaped[:, 0] # Use the first predicted step for simplicity
lstm_predictions_mg_dl = scaler.inverse_transform(dummy_lstm_predictions)[:, 0]

# Inverse transform the true values (y_test) - already done for the Transformer model, can reuse
# y_test_mg_dl = scaler.inverse_transform(y_test.reshape(-1, OUTPUT_WINDOW))[:, 0]

# Visualize LSTM predictions vs. true values
print("\n--- Visualizing LSTM Predictions vs. True Values ---")

plt.figure(figsize=(15, 6))
plt.plot(y_test_mg_dl, color='blue', label='True Glucose Values')
plt.plot(lstm_predictions_mg_dl, color='green', label='LSTM Predicted Glucose Values', alpha=0.7)
plt.title('LSTM Glucose Prediction vs. Actual Values (Test Set)')
plt.xlabel('Time Steps (in 5-minute intervals)')
plt.ylabel('Glucose (mg/dL)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print("\n--- Visualizing Transformer and LSTM Predictions vs. True Values ---")

plt.figure(figsize=(15, 6))
plt.plot(y_test_mg_dl, color='blue', label='True Glucose Values')
plt.plot(predictions_mg_dl, color='red', label='Transformer Predicted Glucose Values', alpha=0.7)
plt.plot(lstm_predictions_mg_dl, color='green', label='LSTM Predicted Glucose Values', alpha=0.7)

plt.title('Transformer and LSTM Glucose Prediction vs. Actual Values (Test Set)')
plt.xlabel('Time Steps (in 5-minute intervals)')
plt.ylabel('Glucose (mg/dL)')
plt.legend()
plt.grid(True)
plt.show()